# Error Analysis - SemEval 2026 Task 4

Qualitative analysis of misclassified pairs to understand model weaknesses.

In [ ]:
import os
import json
import numpy as np
from typing import List, Tuple
from tqdm import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def load_dev_triples(path: str) -> List[Tuple[str, str, str, bool]]:
    triples = []
    for o in read_jsonl(path):
        if {"anchor_text", "text_a", "text_b", "text_a_is_closer"} <= set(o.keys()):
            triples.append((o["anchor_text"], o["text_a"], o["text_b"], bool(o["text_a_is_closer"])))
    return triples

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Load Model & Dev Data

In [ ]:
MODEL_DIR = "../checkpoints/track_a"  # or path to your trained model
DEV_PATH = "../data/raw/dev_track_a.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
backbone = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True).to(device)
backbone.eval()

dev_triples = load_dev_triples(DEV_PATH)
print(f"Loaded {len(dev_triples)} dev triples")

## 2. Run Evaluation & Collect Errors

In [ ]:
@torch.no_grad()
def get_emb(text, max_len=512):
    tokens = tokenizer(text, truncation=True, padding='max_length',
                       max_length=max_len, return_tensors='pt').to(device)
    out = backbone(**tokens)
    emb = out.last_hidden_state.mean(dim=1)
    return F.normalize(emb, p=2, dim=1).squeeze(0)

correct_samples = []
error_samples = []

for i, (anchor, a, b, label) in enumerate(tqdm(dev_triples, desc="Evaluating")):
    va = get_emb(anchor)
    vA = get_emb(a)
    vB = get_emb(b)

    simA = torch.dot(va, vA).item()
    simB = torch.dot(va, vB).item()
    predicted = simA > simB

    record = {
        "idx": i,
        "anchor": anchor[:200],
        "text_a": a[:200],
        "text_b": b[:200],
        "label": label,
        "predicted": predicted,
        "simA": simA,
        "simB": simB,
        "margin": abs(simA - simB),
    }

    if (label and predicted) or (not label and not predicted):
        correct_samples.append(record)
    else:
        error_samples.append(record)

acc = len(correct_samples) / len(dev_triples)
print(f"\nAccuracy: {acc:.4f} ({len(correct_samples)}/{len(dev_triples)})")
print(f"Errors: {len(error_samples)}")

## 3. Analyze Error Patterns

In [ ]:
# Sort errors by confidence (margin) — low margin = model was uncertain
error_samples.sort(key=lambda x: x["margin"])

print("=" * 70)
print("TOP 10 ERRORS (sorted by margin — lowest first)")
print("=" * 70)

for e in error_samples[:10]:
    print(f"\n--- Sample {e['idx']} | margin={e['margin']:.4f} | simA={e['simA']:.4f} simB={e['simB']:.4f} ---")
    print(f"  Label: text_a_is_closer={e['label']} | Predicted: {e['predicted']}")
    print(f"  Anchor: {e['anchor'][:100]}...")
    print(f"  Text A: {e['text_a'][:100]}...")
    print(f"  Text B: {e['text_b'][:100]}...")

## 4. Margin Distribution

In [ ]:
correct_margins = [s["margin"] for s in correct_samples]
error_margins = [s["margin"] for s in error_samples]

print(f"Correct predictions — mean margin: {np.mean(correct_margins):.4f}, "
      f"median: {np.median(correct_margins):.4f}")
print(f"Wrong predictions   — mean margin: {np.mean(error_margins):.4f}, "
      f"median: {np.median(error_margins):.4f}")

# Optional: plot histogram if matplotlib available
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(correct_margins, bins=30, alpha=0.6, label="Correct")
    ax.hist(error_margins, bins=30, alpha=0.6, label="Error")
    ax.set_xlabel("Similarity margin |simA - simB|")
    ax.set_ylabel("Count")
    ax.set_title("Margin distribution: correct vs error predictions")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## 5. Error Categories (Manual Annotation Helper)

In [ ]:
# Print full text of worst errors for manual qualitative analysis
print("=" * 70)
print("FULL TEXT OF TOP 5 ERRORS FOR QUALITATIVE ANALYSIS")
print("=" * 70)

for e in error_samples[:5]:
    triple = dev_triples[e["idx"]]
    print(f"\n{'='*70}")
    print(f"Sample {e['idx']} | simA={e['simA']:.4f} simB={e['simB']:.4f} | Label={e['label']}")
    print(f"{'='*70}")
    print(f"\nANCHOR:\n{triple[0][:500]}")
    print(f"\nTEXT A (label={'closer' if e['label'] else 'farther'}):\n{triple[1][:500]}")
    print(f"\nTEXT B (label={'farther' if e['label'] else 'closer'}):\n{triple[2][:500]}")